# Phát hiện lừa đảo bằng nội dung trang và độ bền trước tin nhắn viết lại: làm lại từng bước và kiểm mọi con số

Notebook này làm lại toàn bộ thí nghiệm của bài về hai câu hỏi:

- trên các trang lừa đảo nhắm tới người Việt, ghép đặc trưng URL vào nội dung trang (text, JavaScript) có giúp bộ phát hiện không, và kết luận có đổi theo
  bộ mã hoá văn bản (TF-IDF, PhoBERT, PhoBERT-v2, ViSoBERT, XLM-R) không;
- bộ phát hiện tin nhắn lừa đảo bị qua mặt tới đâu khi tin nhắn được một LLM viết lại (paraphrase), và huấn luyện đối kháng có cứu được không.

Mọi phép tính nằm ngay trong notebook. Notebook chỉ chạy, in kết quả, và so kết quả với bản kỳ vọng mà bài đã dùng.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `content_paraphrase_walkthrough.ipynb`, `expected/` và `requirements.txt`).
2. Cài thư viện: `pip install -r requirements.txt` (Python 3.12). Các bước dùng mô hình ngôn ngữ chỉ cho kết quả trùng khớp với đúng phiên bản thư viện đã ghim.
3. Tải gói dữ liệu của bài từ Zenodo (DOI ở ô đầu tiên) và giải nén vào thư mục `content_data/` cạnh notebook, hoặc để ô đầu tự tải.
4. Mở notebook và chạy hết (Run All).

Ngoài gói Zenodo, notebook tự tải hai thứ từ Hugging Face, đều ghim theo revision: bộ tin nhắn SMS thật (Bước 8) và bốn bộ mã hoá văn bản (Bước 13,
khoảng 2,5 GB). Bước 1 tới 12 chạy vài phút; Bước 13 tới 17 dùng mô hình ngôn ngữ và mất khoảng hai tiếng trên CPU, phần lớn ở Bước 14, 16 và 17.

### Gói dữ liệu gồm gì

| File | Nội dung |
|---|---|
| `synthetic_sms.csv`, `synthetic_email.csv` | corpus tin nhắn mô phỏng, cả hai lớp do cùng một LLM viết: tin lừa đảo theo yêu cầu viết tin dụ, tin hợp lệ theo yêu cầu viết thông báo dịch vụ; mọi đường link là link giả `sim.example.vn` |
| `paraphrase.csv` | bản viết lại của từng tin lừa đảo, độ mạnh không kiểm soát; biến thể `a` dùng để huấn luyện, `b` dùng để tấn công |
| `paraphrase_band.csv` | bản viết lại có độ mạnh kiểm soát: độ trùng từ Jaccard với tin gốc nằm trong [0,20; 0,30] |
| `paraphrase_pilot_ids.csv` | 73 tin gốc thuộc đợt viết lại đầu tiên (pilot); phần còn lại là đợt mở rộng |
| `pages.csv` | 1.919 trang tiếng Việt đã crawl: text hiển thị, vector đặc trưng URL (21 cột `url_*`) và vector JavaScript (cột `js_*`), theo đúng thứ tự gốc. Không có HTML thô hay ảnh chụp |
| `human_eval_pairs.csv`, `human_eval_ratings.csv` | 50 cặp (tin gốc, bản viết lại) và điểm của hai người chấm |

### Notebook kiểm gì

Mỗi bước so kết quả của mình với file kết quả của bài trong `expected/`: file JSON so từng trường, file CSV so từng byte. Dòng `[bước] x/y checks match`
cho biết bao nhiêu phép so khớp; bước cuối gom tất cả.

| Bước | Làm gì |
|---|---|
| 0 | thiết lập, tải dữ liệu, các hàm kiểm và hàm thống kê dùng chung |
| 1 | corpus tin nhắn mô phỏng |
| 2 | làm rối ký tự và huấn luyện đối kháng |
| 3 | tấn công viết lại, độ mạnh không kiểm soát |
| 4 | tấn công viết lại, độ mạnh kiểm soát |
| 5 | đường liều lượng: độ mạnh viết lại và tỷ lệ lọt |
| 6 | kênh phụ: độ dài tin nhắn |
| 7 | bỏ một kịch bản ra ngoài |
| 8 | thử trên tin nhắn SMS thật |
| 9 | chấm điểm thủ công các bản viết lại |
| 10 | các trang web: thời điểm chụp và tập con cân bằng |
| 11 | ghép kênh trên trang web, TF-IDF |
| 12 | độ nhạy theo tầng nhãn |
| 13 | bộ mã hoá văn bản: tải và nạp |
| 14 | quét năm bộ mã hoá |
| 15 | kiểm định tương đương (TOST) |
| 16 | XLM-R với đầu tuyến tính và đầu XGBoost |
| 17 | chuẩn hoá thang đo và ghép muộn |
| 18 | tổng hợp |

## Bước 0: Thiết lập và tải dữ liệu

Ô đầu khai báo nguồn dữ liệu và tải gói Zenodo nếu thư mục `content_data/` chưa có, rồi kiểm SHA-256 của từng file theo `SHA256SUMS` đi kèm gói.

In [ ]:
import os, re, io, csv, json, math, random, hashlib, zipfile, filecmp, itertools, unicodedata, urllib.request, warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy import sparse, stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score, recall_score, average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

ZENODO_DOI = "10.5281/zenodo.XXXXXXX"
ZENODO_RECORD = None  # the record number, e.g. 1234567; filled in when the deposit is published
HF_SMS = ("trannguyenthaituan/vietnamese_sms_dataset", "a90e2bd7e8df2376939658075ba8094dcee488ad")
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/content_paraphrase/expected"

HOME = Path.cwd()
DATA = HOME / "content_data"
WORK = HOME / "content_work"
WORK.mkdir(exist_ok=True)
OUT = WORK / "results"
OUT.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

def fetch(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PhishVN walkthrough notebook)"})
    Path(dest).write_bytes(urllib.request.urlopen(req, timeout=300).read())

if not (DATA / "SHA256SUMS").exists():
    assert ZENODO_RECORD, f"download the data package (DOI {ZENODO_DOI}) and unzip it into {DATA}"
    DATA.mkdir(exist_ok=True)
    meta = json.loads(urllib.request.urlopen(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=120).read())
    for f in meta["files"]:
        fetch(f["links"]["self"], DATA / f["key"])
for line in (DATA / "SHA256SUMS").read_text().splitlines():
    digest, name = line.split()
    assert sha(DATA / name) == digest, f"{name} differs from the published package"

EXPECTED = HOME / "expected"
EXPECTED_FILES = json.loads((EXPECTED / "manifest.json").read_text()) if (EXPECTED / "manifest.json").exists() else None
if EXPECTED_FILES is None:
    EXPECTED = WORK / "expected"
    (EXPECTED / "results").mkdir(parents=True, exist_ok=True)
    fetch(f"{EXPECTED_URL}/manifest.json", EXPECTED / "manifest.json")
    EXPECTED_FILES = json.loads((EXPECTED / "manifest.json").read_text())
    for f in EXPECTED_FILES:
        if not (EXPECTED / f).exists():
            fetch(f"{EXPECTED_URL}/{f}", EXPECTED / f)
print("data:", DATA, "| expected:", EXPECTED, f"({len(EXPECTED_FILES)} files)")

Ô tiếp theo định nghĩa các hàm kiểm: `check` ghi một phép so, `report` in số phép khớp của một bước, `check_file` so từng byte một file với bản kỳ vọng, và
`check_json` so mọi trường của một kết quả với file JSON kỳ vọng, hoặc với một mục của file đó (in ra những trường lệch, nếu có).

In [ ]:
CHECKS = []

def _same(a, b):
    if isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b):
        return True
    return a == b

def check(step, name, mine, ref):
    ok = _same(mine, ref)
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": ref, "match": bool(ok)})
    return ok

def report(step):
    rows = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows if not c["match"]]
    print(f"[{step}] {len(rows) - len(bad)}/{len(rows)} checks match")
    for c in bad[:10]:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

def check_file(step, produced, expected_name):
    check(step, f"{expected_name} byte-identical", filecmp.cmp(produced, EXPECTED / expected_name, shallow=False), True)

def _plain(o):
    # the same conversion the saved results went through: numpy scalars to Python, tuples to lists
    def conv(v):
        if hasattr(v, "item"):
            return v.item()
        if hasattr(v, "tolist"):
            return v.tolist()
        return str(v)
    return json.loads(json.dumps(o, sort_keys=True, default=conv, allow_nan=True))

def flat(d, prefix=""):
    if isinstance(d, dict):
        for k, v in d.items():
            yield from flat(v, f"{prefix}.{k}" if prefix else str(k))
    elif isinstance(d, list):
        for i, v in enumerate(d):
            yield from flat(v, f"{prefix}[{i}]")
    else:
        yield prefix, d

def check_json(step, mine, saved_name, key=None):
    # key: compare against one top-level entry of the file only
    saved = json.loads((EXPECTED / saved_name).read_text())
    new = dict(flat(_plain(mine)))
    ref = dict(flat(saved if key is None else saved[key]))
    keys = sorted(set(new) | set(ref))
    diff = [k for k in keys if not _same(new.get(k, "<missing>"), ref.get(k, "<missing>"))]
    check(step, f"{saved_name}{'[' + key + ']' if key else ''}: fields that differ (of {len(keys)})", len(diff), 0)
    for k in diff[:10]:
        print(f"    differs: {k}: saved={ref.get(k, '<missing>')!r} notebook={new.get(k, '<missing>')!r}")

Hàm thống kê dùng chung cho cả bài:

- `corrected_paired_t`: so hai cấu hình trên cùng các lần chia train/test. Vì các tập test của các lần chia chồng lên nhau, sai số chuẩn được nới theo
  Nadeau và Bengio: `se = sqrt((1/K + n_test/n_train) * Var(d))`, với `d` là hiệu của từng lần chia. Hàm trả về hiệu trung bình, p đã hiệu chỉnh, p chưa hiệu
  chỉnh, và số lần chia nghiêng về cấu hình thứ nhất.
- `bh_adjust`: hiệu chỉnh Benjamini-Hochberg cho một họ kiểm định.
- `wilson`: khoảng tin cậy Wilson cho một tỷ lệ.

In [ ]:
DEFAULT_TEST_FRAC = 0.30


def corrected_paired_t(d, test_frac: float = DEFAULT_TEST_FRAC) -> dict:
    d = np.asarray(d, dtype=float)
    k = len(d)
    mean = float(np.mean(d))
    var = float(np.var(d, ddof=1)) if k > 1 else 0.0
    out = {"k": k, "mean": mean, "wins": int(np.sum(d > 0)),
           "p": 1.0, "p_naive": 1.0, "se": 0.0, "inflation": float("nan")}
    if k < 2 or var <= 0:
        return out
    try:
        from scipy import stats
    except ImportError:  # keep the pipeline runnable without scipy; verdicts fall back to "no evidence"
        return out
    factor = 1.0 / k + test_frac / (1.0 - test_frac)
    se = math.sqrt(factor * var)
    se_naive = math.sqrt(var / k)
    t = mean / se
    out.update(p=float(2 * stats.t.sf(abs(t), k - 1)),
               p_naive=float(2 * stats.t.sf(abs(mean / se_naive), k - 1)),
               se=se, t=float(t), inflation=se / se_naive)
    return out


def wilson(k: float, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))



def bh_adjust(pvals, alpha=0.05):
    idx = sorted(range(len(pvals)), key=lambda i: pvals[i])
    m = len(pvals)
    adj = [1.0] * m
    running = 1.0
    for rank in range(m, 0, -1):          # walk up from the largest p-value
        i = idx[rank - 1]
        running = min(running, pvals[i] * m / rank)
        adj[i] = min(1.0, running)
    return adj, [a <= alpha for a in adj]

## Bước 1: Corpus tin nhắn mô phỏng

Corpus gồm tin SMS và email, cả hai lớp do cùng một LLM (`claude-fable-5`) viết: tin lừa đảo theo yêu cầu viết tin dụ, trải trên bảy kịch bản (ngân hàng, giao
hàng, thương mại điện tử, cơ quan nhà nước, mạng xã hội, thuế, viễn thông); tin hợp lệ theo yêu cầu viết thông báo dịch vụ (biến động số dư, giao hàng, lịch hẹn).
Không tin nào do người viết hay lấy từ lưu lượng thật, nên kết quả ở Bước 2 tới 7 là hành vi bên trong văn phong của một mô hình; Bước 8 thử trên tin thật. Trước mọi phân tích, mọi đường link bị xoá khỏi văn bản (`strip_url`), để bộ phát hiện chỉ có chữ mà dựa vào.

`load_corpus` nạp hai file, xoá link, gán nhãn `y` (1 là lừa đảo) và bỏ tin rỗng. Nếu truyền vào một bảng viết lại, hàm gắn thêm hai cột: `para_a` (biến thể dùng
để huấn luyện) và `para_b` (biến thể dùng để tấn công) cho mỗi tin lừa đảo.

In [ ]:
URL_RE = re.compile(r"https?://\S+|\bsim\.example\.vn\S*", re.I)


SEEDS = 20


CELLS = ("D0_A0", "D0_A1", "D0_A2", "D1_A0", "D1_A1", "D1_A2", "D2_A0", "D2_A2")


def strip_url(s: str) -> str:
    return URL_RE.sub(" ", str(s))


def vec():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1, max_features=20000)


def miss(clf, v, texts, y_true) -> float:
    pred = clf.predict(v.transform(texts))
    return 1.0 - recall_score(y_true, pred, pos_label=1, zero_division=0)



def load_corpus(para_file=None):
    df = pd.concat([pd.read_csv(DATA / f"synthetic_{t}.csv") for t in ("sms", "email")], ignore_index=True)
    df["text"] = df["text"].fillna("").astype(str).map(strip_url)
    df["y"] = (df["label"] == "phishing").astype(int)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    if para_file is None:
        return df
    para = pd.read_csv(DATA / para_file)
    para["text"] = para["text"].map(strip_url)
    wide = para.pivot(index="src_id", columns="variant", values="text")
    assert not set(df.loc[df.y == 1, "id"]) - set(wide.index), "every lure needs a rewrite"
    df["para_a"] = df["id"].map(wide["a"]).fillna("")
    df["para_b"] = df["id"].map(wide["b"]).fillna("")
    return df, para


corpus = load_corpus()
per_channel = {ch: (int(sub["y"].sum()), int((sub["y"] == 0).sum()))
               for ch in ("sms", "email") for sub in [corpus[corpus["channel"].astype(str).str.lower() == ch]]}
print(f"{int(corpus.y.sum())} phishing / {int((corpus.y == 0).sum())} benign; per channel (phishing, benign): {per_channel}")
display(corpus.groupby(["scenario", "label"]).size().unstack(fill_value=0))

## Bước 2: Làm rối ký tự và huấn luyện đối kháng

Kiểu tấn công đã được công bố từ trước: chèn ký tự vô hình, thay chữ Latin bằng chữ Kirin trông giống hệt, và đổi chữ thành số (`perturb`). Bộ phát hiện là hồi
quy logistic trên n-gram ký tự (TF-IDF). Với 20 lần chia train/test (70/30), bước này đo tỷ lệ tin lừa đảo bị lọt khi tin còn sạch, khi tin bị làm rối, và khi
bộ phát hiện đã được huấn luyện thêm trên tin bị làm rối (mỗi tin lừa đảo trong tập huấn luyện thêm hai bản rối). Hai phép so có cặp: làm rối có tăng tỷ lệ lọt
không, và huấn luyện đối kháng có kéo nó xuống không.

In [ ]:
HOMO = {"a": "а", "e": "е", "o": "о", "c": "с", "p": "р", "i": "і"}  # latin->cyrillic look-alikes


def perturb(text: str, rng: random.Random) -> str:
    s = list(text)
    for i in range(len(s)):
        r = rng.random()
        if r < 0.05:                                  # zero-width space
            s[i] = s[i] + "​"
        elif r < 0.08 and s[i].lower() in HOMO:       # homoglyph
            s[i] = HOMO[s[i].lower()]
        elif r < 0.10 and s[i].isalpha():             # leetspeak
            s[i] = {"a": "4", "e": "3", "o": "0", "i": "1"}.get(s[i].lower(), s[i])
    return "".join(s)


def llm_run():
    df = load_corpus()
    texts = df["text"].to_numpy(); y = df["y"].to_numpy()
    res = {k: [] for k in ("f1_clean", "miss_clean", "miss_obf", "miss_obf_adv")}
    for s in range(SEEDS):
        rng = random.Random(s)
        tr, te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=y, random_state=s)
        Xtr_txt, ytr = texts[tr], y[tr]
        Xte_txt, yte = texts[te], y[te]
        ph = yte == 1
        # perturbed (obfuscated) copies of the held-out phishing
        obf_txt = np.array([perturb(t, rng) if ph[i] else t for i, t in enumerate(Xte_txt)], dtype=object)

        # baseline detector (no adversarial training)
        v = vec(); Xtr = v.fit_transform(Xtr_txt)
        clf = LogisticRegression(max_iter=3000, class_weight="balanced").fit(Xtr, ytr)
        pc = clf.predict(v.transform(Xte_txt))
        po = clf.predict(v.transform(obf_txt))
        res["f1_clean"].append(f1_score(yte, pc, zero_division=0))
        res["miss_clean"].append(1 - recall_score(yte, pc, pos_label=1, zero_division=0))
        res["miss_obf"].append(1 - recall_score(yte, po, pos_label=1, zero_division=0))

        # adversarially-trained detector: augment training phishing with perturbed copies
        aug_txt = list(Xtr_txt) + [perturb(t, rng) for t, yy in zip(Xtr_txt, ytr) if yy == 1 for _ in range(2)]
        aug_y = list(ytr) + [1] * (2 * int(ytr.sum()))
        va = vec(); Xtra = va.fit_transform(aug_txt)
        clfa = LogisticRegression(max_iter=3000, class_weight="balanced").fit(Xtra, aug_y)
        poa = clfa.predict(va.transform(obf_txt))
        res["miss_obf_adv"].append(1 - recall_score(yte, poa, pos_label=1, zero_division=0))
    return df, res


CONTRASTS = (
    ("evasion", "miss_obf", "miss_clean",
     "obfuscation raises the naive detector's miss rate"),
    ("repair", "miss_obf", "miss_obf_adv",
     "adversarial training lowers the miss rate on the same obfuscated messages"),
)


def llm_tests(res: dict) -> dict:
    out = {}
    for key, hi, lo, label in CONTRASTS:
        d = np.asarray(res[hi], float) - np.asarray(res[lo], float)
        out[key] = {**corrected_paired_t(d), "label": label}
    adj, rej = bh_adjust([out[k]["p"] for k, *_ in CONTRASTS])
    for (key, *_), a, rj in zip(CONTRASTS, adj, rej):
        out[key]["p_adj"], out[key]["reject"] = a, bool(rj)
    return out



df_llm, res_llm = llm_run()
T_llm = llm_tests(res_llm)
gens = sorted(df_llm.loc[df_llm["y"] == 1, "gen_model"].dropna().astype(str).unique())
for key, hi, lo, label in CONTRASTS:
    t = T_llm[key]
    print(f"{key:8s} {label}: delta {t['mean'] * 100:+.2f} pp, {t['wins']}/{t['k']} splits, corrected p={t['p']:.4f}, BH q={t['p_adj']:.4f}")
check_json("2 obfuscation", {"n_phishing": int(df_llm.y.sum()), "n_benign": int((df_llm.y == 0).sum()), "per_channel": per_channel,
                             "generators": gens, "per_split": res_llm, "tests": T_llm}, "results/llm_corpus.json")
report("2 obfuscation")

## Bước 3: Tấn công viết lại, độ mạnh không kiểm soát

Ba bộ phát hiện: D0 không phòng thủ; D1 huấn luyện đối kháng trên tin làm rối ký tự (cách phòng thủ đã công bố); D2 huấn luyện đối kháng trên bản viết lại
(biến thể `a`). Mỗi bộ được thử trên tin lừa đảo sạch (A0), tin làm rối (A1) và bản viết lại biến thể `b` (A2); D2 không được thử trên A1 vì không giả thuyết nào
cần ô đó. Số đo là tỷ lệ lọt trên tin lừa đảo của tập test, cùng tỷ lệ báo nhầm trên tin hợp lệ.

Năm phép so đã đăng ký trước (H1, H1b, H2, H3 và một phép đối chứng) dùng `corrected_paired_t`, rồi hiệu chỉnh BH. `corpus_multiple_for_significance` ước tính
corpus phải lớn gấp bao nhiêu lần để hiệu đo được đạt ý nghĩa thống kê. `lexical_shift` đo độ trùng từ (Jaccard) giữa tin gốc và bản viết lại, cho toàn bộ và
riêng từng đợt viết lại.

In [ ]:
def para_run(df: pd.DataFrame):
    txt, y = df["text"].to_numpy(), df["y"].to_numpy()
    pa, pb = df["para_a"].to_numpy(), df["para_b"].to_numpy()
    per_seed = {c: [] for c in CELLS}
    fpr = {"D0": [], "D1": [], "D2": []}

    for s in range(SEEDS):
        rng = random.Random(s)
        tr, te = train_test_split(np.arange(len(df)), test_size=0.30, stratify=y, random_state=s)
        ytr, yte = y[tr], y[te]
        ph_te = te[yte == 1]                       # held-out phishing, attacked in A1/A2
        be_te = te[yte == 0]                       # benign, never attacked
        y_ph = np.ones(len(ph_te), dtype=int)

        clean_ph = txt[ph_te]
        obf_ph = np.array([perturb(t, rng) for t in clean_ph], dtype=object)
        para_ph = pb[ph_te]                        # variant 'b' — test role only

        # D0: naive
        v0 = vec(); clf0 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
            v0.fit_transform(txt[tr]), ytr)
        # D1: adversarially trained on character obfuscation (the published defence)
        tr_ph_txt = txt[tr][ytr == 1]
        aug1 = list(txt[tr]) + [perturb(t, rng) for t in tr_ph_txt for _ in range(2)]
        y1 = list(ytr) + [1] * (2 * len(tr_ph_txt))
        v1 = vec(); clf1 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
            v1.fit_transform(aug1), y1)
        # D2: adversarially trained on paraphrases, variant 'a' only (train role)
        aug2 = list(txt[tr]) + [t for t in pa[tr][ytr == 1] if t]
        y2 = list(ytr) + [1] * int(sum(1 for t in pa[tr][ytr == 1] if t))
        v2 = vec(); clf2 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
            v2.fit_transform(aug2), y2)

        per_seed["D0_A0"].append(miss(clf0, v0, clean_ph, y_ph))
        per_seed["D0_A1"].append(miss(clf0, v0, obf_ph, y_ph))
        per_seed["D0_A2"].append(miss(clf0, v0, para_ph, y_ph))
        per_seed["D1_A0"].append(miss(clf1, v1, clean_ph, y_ph))
        per_seed["D1_A1"].append(miss(clf1, v1, obf_ph, y_ph))
        per_seed["D1_A2"].append(miss(clf1, v1, para_ph, y_ph))
        per_seed["D2_A0"].append(miss(clf2, v2, clean_ph, y_ph))
        per_seed["D2_A2"].append(miss(clf2, v2, para_ph, y_ph))
        for name, (c, v) in (("D0", (clf0, v0)), ("D1", (clf1, v1)), ("D2", (clf2, v2))):
            fpr[name].append(float(np.mean(c.predict(v.transform(txt[be_te])) == 1)))

    return per_seed, fpr


COMPARISONS = (
    ("H1",   "D0_A2", "D0_A0", "paraphrase vs clean, naive detector", +1),
    ("H1b",  "D0_A2", "D0_A1", "paraphrase vs character obfuscation, naive detector", +1),
    ("H2",   "D1_A2", "D0_A2", "char-adversarial training against the paraphrase attack", +1),
    ("H3",   "D2_A2", "D0_A2", "paraphrase-adversarial training against the paraphrase attack", -1),
    ("ctrl", "D0_A1", "D0_A0", "character obfuscation vs clean (replicates the published cell)", +1),
)


def corpus_multiple_for_significance(t_now: float, k: int = SEEDS, alpha: float = 0.05):
    try:
        from scipy import stats
    except ImportError:
        return float("nan")
    if not t_now or t_now != t_now:
        return float("nan")
    t_req = float(stats.t.ppf(1 - alpha / 2, k - 1))
    return float((t_req / abs(t_now)) ** 2)


def lexical_shift(df: pd.DataFrame) -> dict:

    ph = df[df.y == 1]
    out: dict = {}
    for name, mask in (("all", ph["id"].notna()),
                       ("pilot", ph["id"].isin(PILOT_IDS)),
                       ("extension", ~ph["id"].isin(PILOT_IDS))):
        sub = ph[mask]
        js = []
        for src, para in zip(sub["text"], sub["para_b"]):
            a, b = set(str(src).lower().split()), set(str(para).lower().split())
            if a or b:
                js.append(len(a & b) / len(a | b))
        out[name] = (float(np.mean(js)) if js else float("nan"), len(js))
    return out



PILOT_IDS = set(pd.read_csv(DATA / "paraphrase_pilot_ids.csv")["src_id"])


def paired_tests(per_seed, para_style):
    # the registered comparisons, corrected-t then BH; the two studies count `consistent` slightly differently
    tests = []
    for key, hi, lo, label, direction in COMPARISONS:
        d = np.array(per_seed[hi]) - np.array(per_seed[lo])
        r = corrected_paired_t(d)
        consistent = int(np.sum(d > 0)) if direction > 0 else int(np.sum(d < 0))
        t = {"key": key, "hi": hi, "lo": lo, "label": label}
        if para_style:
            t["direction"] = direction
        t.update({"consistent": consistent, "need": corpus_multiple_for_significance(r.get("t", float("nan"))), **r})
        tests.append(t)
    adj, rej = bh_adjust([t["p"] for t in tests])
    for t, a, rj in zip(tests, adj, rej):
        t["p_adj"], t["reject"] = a, rj
    return tests


df_para, _ = load_corpus("paraphrase.csv")
per_seed, fpr = para_run(df_para)
tests = paired_tests(per_seed, para_style=True)
T_para = {t["key"]: t for t in tests}
jac = lexical_shift(df_para)
mean = {c: float(np.mean(per_seed[c])) for c in CELLS}
sd = {c: float(np.std(per_seed[c])) for c in CELLS}

pd.DataFrame([{"key": t["key"], "hi": t["hi"], "lo": t["lo"], "label": t["label"],
               "mean_pp": t["mean"] * 100, "p": t["p"], "p_naive": t["p_naive"],
               "p_adj": t["p_adj"], "reject": int(t["reject"]),
               "consistent": t["consistent"], "k": SEEDS,
               "corpus_multiple_needed": t["need"]} for t in tests]).to_csv(OUT / "p3_paraphrase_stats.csv", index=False)
pd.DataFrame([{"cell": c, "miss_mean_pp": mean[c] * 100, "miss_sd_pp": sd[c] * 100}
              for c in CELLS]).to_csv(OUT / "p3_paraphrase_cells.csv", index=False)

print("miss rate (%) by detector and attack:")
for c in CELLS:
    print(f"    {c}: {mean[c] * 100:5.1f} +- {sd[c] * 100:.1f}")
for t in tests:
    print(f"    {t['key']:5s} {t['label']:<62s} delta={t['mean'] * 100:+6.1f}pp  p={t['p']:.4f}  q={t['p_adj']:.4f}")
for k, (v, n) in jac.items():
    print(f"    mean token Jaccard(source, rewrite) [{k}, n={n}]: {v:.3f}")
check_json("3 paraphrase", {"n_phishing": int(df_para.y.sum()), "n_benign": int((df_para.y == 0).sum()),
                            "miss_per_split": per_seed, "benign_fpr_per_split": fpr, "lexical_shift": jac, "tests": T_para},
           "results/paraphrase.json")
check_file("3 paraphrase", OUT / "p3_paraphrase_stats.csv", "p3_paraphrase_stats.csv")
check_file("3 paraphrase", OUT / "p3_paraphrase_cells.csv", "p3_paraphrase_cells.csv")
report("3 paraphrase")

## Bước 4: Tấn công viết lại, độ mạnh kiểm soát

Ở Bước 3, độ mạnh của bản viết lại tuỳ LLM: có bản đổi gần hết từ, có bản gần như giữ nguyên. Bước này lặp lại đúng thiết kế đó trên bộ viết lại thứ hai, trong
đó mọi bản viết lại có độ trùng từ Jaccard với tin gốc nằm trong dải đã đăng ký trước [0,20; 0,30]. Ô dưới kiểm lại điều kiện đó trước khi phân tích.

In [ ]:
BAND = (0.20, 0.30)


def toks(s: str) -> set[str]:
    return set(URL_RE.sub(" ", str(s)).lower().split())


def jaccard(a: str, b: str) -> float:
    A, B = toks(a), toks(b)
    return len(A & B) / len(A | B) if (A | B) else 0.0



df_band, para_band = load_corpus("paraphrase_band.csv")
assert ((para_band["jaccard"] >= BAND[0]) & (para_band["jaccard"] <= BAND[1])).all(), "out-of-band rewrite"
per_seed_b, fpr_b = para_run(df_band)
tests_b = paired_tests(per_seed_b, para_style=False)
T_band = {t["key"]: t for t in tests_b}
pd.DataFrame([{"key": t["key"], "label": t["label"], "mean_pp": t["mean"] * 100,
               "p": t["p"], "p_naive": t["p_naive"], "p_adj": t["p_adj"],
               "reject": int(t["reject"]), "consistent": t["consistent"], "k": SEEDS,
               "corpus_multiple_needed": t["need"]} for t in tests_b]).to_csv(OUT / "p3_band_stats.csv", index=False)
for c in CELLS:
    print(f"    {c}: {np.mean(per_seed_b[c]) * 100:5.1f} +- {np.std(per_seed_b[c]) * 100:.1f}")
for t in tests_b:
    print(f"    {t['key']:5s} delta={t['mean'] * 100:+6.1f}pp  p={t['p']:.4f}  q={t['p_adj']:.4f}  reject={t['reject']}")
j_all, j_test = para_band["jaccard"], para_band.loc[para_band.variant == "b", "jaccard"]
print(f"mean Jaccard: all rewrites {j_all.mean():.3f}, attack variant {j_test.mean():.3f}")
check_json("4 band", {"n_phishing": int(df_band.y.sum()), "miss_per_split": per_seed_b, "benign_fpr_per_split": fpr_b,
                      "tests": T_band, "jaccard_mean_all": float(j_all.mean()), "jaccard_mean_test_variant": float(j_test.mean())},
           "results/paraphrase_band.json")
check_file("4 band", OUT / "p3_band_stats.csv", "p3_band_stats.csv")
report("4 band")

## Bước 5: Đường liều lượng

Bản viết lại càng ít trùng từ với tin gốc thì có càng dễ lọt không? Bước này chạy lại vòng lặp của Bước 3 nhưng ghi lại kết quả của từng bản viết lại ở điều
kiện A2 (`per_rewrite`), gộp về mỗi tin gốc một dòng (`aggregate`), rồi:

- đo tương quan Spearman giữa Jaccard và tỷ lệ lọt cho mỗi bộ phát hiện, và lấy hướng tương quan làm hướng cho đường hồi quy đơn điệu (isotonic, có trọng số);
- so độ dốc của đường trong dải [0,20; 0,30] với độ dốc trung bình trên toàn miền (`steepness`), và tỷ lệ lần bootstrap theo tin gốc mà dải dốc hơn;
- ước lượng tỷ lệ lọt tại giữa dải, tại hai phân vị 10% và 90% của Jaccard, và mức lọt trung bình trên bộ viết lại có kiểm soát, kèm khoảng bootstrap;
- tách đợt pilot và đợt mở rộng.

Lưu ý `per_rewrite` gọi `perturb` trên tin test trước khi huấn luyện D1 dù không dùng kết quả: bộ sinh số ngẫu nhiên dùng chung, nên thứ tự tiêu thụ đó là một
phần định nghĩa của D1 ở Bước 3.

In [ ]:
DETECTORS = ("D0", "D1", "D2")


BOOT = 600            # cluster-bootstrap replicates


BOOT_SEED = 20260817  # fixed, so the intervals are reproducible


GRID = 201


CI = (2.5, 97.5)


def per_rewrite(df: pd.DataFrame) -> pd.DataFrame:
    txt, y = df["text"].to_numpy(), df["y"].to_numpy()
    pa, pb = df["para_a"].to_numpy(), df["para_b"].to_numpy()
    ids = df["id"].to_numpy()
    rows = []

    for s in range(SEEDS):
        rng = random.Random(s)
        tr, te = train_test_split(np.arange(len(df)), test_size=0.30, stratify=y, random_state=s)
        ytr, yte = y[tr], y[te]
        ph_te = te[yte == 1]

        # Drawn before the D1 augmentation, exactly as in run(): the shared rng makes the
        # consumption ORDER part of the definition of D1.
        clean_ph = txt[ph_te]
        for t in clean_ph:
            perturb(t, rng)
        para_ph = pb[ph_te]                        # variant 'b' — test role only

        v0 = vec()
        clf0 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
            v0.fit_transform(txt[tr]), ytr)
        tr_ph_txt = txt[tr][ytr == 1]
        aug1 = list(txt[tr]) + [perturb(t, rng) for t in tr_ph_txt for _ in range(2)]
        y1 = list(ytr) + [1] * (2 * len(tr_ph_txt))
        v1 = vec()
        clf1 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
            v1.fit_transform(aug1), y1)
        aug2 = list(txt[tr]) + [t for t in pa[tr][ytr == 1] if t]
        y2 = list(ytr) + [1] * int(sum(1 for t in pa[tr][ytr == 1] if t))
        v2 = vec()
        clf2 = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
            v2.fit_transform(aug2), y2)

        for name, clf, v in (("D0", clf0, v0), ("D1", clf1, v1), ("D2", clf2, v2)):
            pred = clf.predict(v.transform(para_ph))
            for i, src_id in enumerate(ids[ph_te]):
                rows.append({"seed": s, "src_id": src_id, "detector": name,
                             "missed": int(pred[i] != 1)})
    return pd.DataFrame(rows)


def aggregate(log: pd.DataFrame, jmap: dict[str, float]) -> pd.DataFrame:
    g = (log.groupby(["detector", "src_id"], as_index=False)
            .agg(n_obs=("missed", "size"), n_missed=("missed", "sum")))
    g["miss_rate"] = g["n_missed"] / g["n_obs"]
    g["jaccard"] = g["src_id"].map(jmap)
    return g.dropna(subset=["jaccard"]).sort_values(["detector", "jaccard"]).reset_index(drop=True)


def direction(sub: pd.DataFrame) -> int:
    from scipy import stats
    if sub["jaccard"].nunique() < 2 or sub["miss_rate"].nunique() < 2:
        return 0
    rho = float(stats.spearmanr(sub["jaccard"], sub["miss_rate"]).statistic)
    return 1 if rho > 0 else (-1 if rho < 0 else 0)


def spearman(sub: pd.DataFrame) -> tuple[float, float]:
    from scipy import stats
    r = stats.spearmanr(sub["jaccard"], sub["miss_rate"])
    return float(r.statistic), float(r.pvalue)


def fit(sub: pd.DataFrame, increasing: bool) -> IsotonicRegression:
    iso = IsotonicRegression(increasing=increasing, out_of_bounds="clip")
    iso.fit(sub["jaccard"].to_numpy(float), sub["miss_rate"].to_numpy(float),
            sample_weight=sub["n_obs"].to_numpy(float))
    return iso


def dose_bootstrap(sub: pd.DataFrame, grid: np.ndarray, increasing: bool,
                      b: int = BOOT, seed: int = BOOT_SEED) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n = len(sub)
    if n < 2:
        return np.full(len(grid), np.nan), np.full(len(grid), np.nan)
    j = sub["jaccard"].to_numpy(float)
    m = sub["miss_rate"].to_numpy(float)
    w = sub["n_obs"].to_numpy(float)
    draws = np.empty((b, len(grid)))
    for k in range(b):
        idx = rng.integers(0, n, n)
        iso = IsotonicRegression(increasing=increasing, out_of_bounds="clip")
        iso.fit(j[idx], m[idx], sample_weight=w[idx])
        draws[k] = iso.predict(grid)
    return np.percentile(draws, CI[0], axis=0), np.percentile(draws, CI[1], axis=0)


def band_slope(iso: IsotonicRegression, lo: float, hi: float) -> float:
    a, b = iso.predict(np.array([lo, hi], dtype=float))
    return abs(float(b - a)) / (hi - lo)


def steepness(iso: IsotonicRegression, jmin: float, jmax: float,
              band: tuple[float, float]) -> dict:
    lo, hi = band
    s_band = band_slope(iso, lo, hi)
    s_all = band_slope(iso, jmin, jmax)
    return {"slope_band": s_band, "slope_all": s_all,
            "ratio": (s_band / s_all) if s_all else float("nan"),
            "drop_band": float(np.diff(iso.predict(np.array([lo, hi])))[0]),
            "drop_all": float(np.diff(iso.predict(np.array([jmin, jmax])))[0])}


def band_steeper_fraction(sub: pd.DataFrame, jmin: float, jmax: float,
                          band: tuple[float, float], increasing: bool,
                          b: int = BOOT, seed: int = BOOT_SEED) -> float:
    rng = np.random.default_rng(seed + 1)
    j = sub["jaccard"].to_numpy(float)
    m = sub["miss_rate"].to_numpy(float)
    w = sub["n_obs"].to_numpy(float)
    n, wins = len(sub), 0
    for _ in range(b):
        idx = rng.integers(0, n, n)
        iso = IsotonicRegression(increasing=increasing, out_of_bounds="clip")
        iso.fit(j[idx], m[idx], sample_weight=w[idx])
        wins += band_slope(iso, *band) > band_slope(iso, jmin, jmax)
    return wins / b


def point_ci(sub: pd.DataFrame, at: float, increasing: bool,
             b: int = BOOT, seed: int = BOOT_SEED) -> tuple[float, float, float]:
    iso = fit(sub, increasing)
    est = float(iso.predict(np.array([at]))[0])
    lo, hi = dose_bootstrap(sub, np.array([at]), increasing, b, seed)
    return est, float(lo[0]), float(hi[0])


def mean_ci(sub: pd.DataFrame, b: int = BOOT, seed: int = BOOT_SEED) -> tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    m = sub["miss_rate"].to_numpy(float)
    w = sub["n_obs"].to_numpy(float)
    est = float(np.average(m, weights=w))
    n = len(sub)
    draws = np.empty(b)
    for k in range(b):
        idx = rng.integers(0, n, n)
        draws[k] = np.average(m[idx], weights=w[idx])
    return est, float(np.percentile(draws, CI[0])), float(np.percentile(draws, CI[1]))



def sources():
    d = pd.concat([pd.read_csv(DATA / f"synthetic_{t}.csv") for t in ("sms", "email")], ignore_index=True)
    d = d[d["label"] == "phishing"]
    return {r["id"]: str(r["text"]) for _, r in d.iterrows()}


def predecessor_jaccard():
    # J between each source lure and its attack-role rewrite, on the raw texts
    src = sources()
    pre = pd.read_csv(DATA / "paraphrase.csv")
    pre = pre[pre["variant"] == "b"]
    return {r.src_id: float(jaccard(src[r.src_id], r.text)) for r in pre.itertuples() if r.src_id in src}


def strata(jm):
    out = {}
    for name, keep in (("pilot", lambda i: i in PILOT_IDS), ("extension", lambda i: i not in PILOT_IDS)):
        vals = [v for i, v in jm.items() if keep(i)]
        out[name] = ((float(np.mean(vals)) if vals else float("nan")), len(vals))
    return out


df_dose = df_para
jm = predecessor_jaccard()
assert not set(df_dose.loc[df_dose.y == 1, "id"]) - set(jm)
log = per_rewrite(df_dose)
agg = aggregate(log, jm)
agg.to_csv(OUT / "p3_dose_response.csv", index=False)

incr = {d: direction(agg[agg.detector == d]) > 0 for d in DETECTORS}
rho = {d: spearman(agg[agg.detector == d]) for d in DETECTORS}
lo, hi = BAND
jmin, jmax = min(jm.values()), max(jm.values())
st = {d: steepness(fit(agg[agg.detector == d], incr[d]), jmin, jmax, BAND) for d in DETECTORS}

bj = {r.src_id: float(r.jaccard) for r in para_band[para_band.variant == "b"].itertuples()}
band_agg = aggregate(per_rewrite(df_band), bj)
band_j = float(para_band["jaccard"].mean())
band_pt = {d: mean_ci(band_agg[band_agg.detector == d]) for d in DETECTORS}
shared = len(set(band_agg[band_agg.detector == "D0"].src_id) & set(agg[agg.detector == "D0"].src_id))

d0 = agg[agg.detector == "D0"]
iso0 = fit(d0, incr["D0"])
at_lo, at_hi = float(iso0.predict(np.array([lo]))[0]), float(iso0.predict(np.array([hi]))[0])
mid = (lo + hi) / 2
mid_est, mid_lo, mid_hi = point_ci(d0, mid, incr["D0"])
q_lo, q_hi = float(np.quantile(d0["jaccard"], 0.10)), float(np.quantile(d0["jaccard"], 0.90))
at_q_lo, at_q_hi = (float(v) for v in iso0.predict(np.array([q_lo, q_hi])))
steep_frac = band_steeper_fraction(d0, jmin, jmax, BAND, incr["D0"])
pooled = float(np.average(d0["miss_rate"], weights=d0["n_obs"]))
strat = strata(jm)
d0s = d0.assign(pilot=d0["src_id"].isin(PILOT_IDS), fitted=iso0.predict(d0["jaccard"].to_numpy(float)))
stratum = {}
for name, flag in (("pilot", True), ("extension", False)):
    t = d0s[d0s["pilot"] == flag]
    stratum[name] = {"n": int(len(t)), "obs": float(np.average(t["miss_rate"], weights=t["n_obs"])),
                     "implied": float(np.average(t["fitted"], weights=t["n_obs"]))}

print(f"J support [{jmin:.3f}, {jmax:.3f}], band [{lo:.2f}, {hi:.2f}]")
for d in DETECTORS:
    print(f"    {d}: Spearman rho={rho[d][0]:+.3f} (p={rho[d][1]:.3g}), band slope / mean slope = {st[d]['ratio']:.2f}")
print(f"D0 miss {at_q_lo * 100:.1f}% at J10={q_lo:.3f} -> {at_q_hi * 100:.1f}% at J90={q_hi:.3f}; "
      f"band steeper in {steep_frac * 100:.0f}% of bootstrap replicates")
for d, (e, plo, phi) in band_pt.items():
    print(f"    controlled-strength corpus {d}: miss {e * 100:.1f}% [{plo * 100:.1f}, {phi * 100:.1f}]")
check_json("5 dose response", {
    "increasing": incr, "spearman": rho, "steepness": st, "j_support": [jmin, jmax],
    "band_point": band_pt, "band_mean_j": band_j, "band_shared_sources": shared,
    "d0_at_band_edges": [at_lo, at_hi], "d0_mid": [mid, mid_est, mid_lo, mid_hi],
    "d0_quantiles": [q_lo, q_hi, at_q_lo, at_q_hi], "band_steeper_fraction": steep_frac,
    "d0_pooled_miss": pooled, "n_observations": int(len(log)), "n_sources": int(d0["src_id"].nunique()),
    "strata": strat, "stratum_fit": stratum}, "results/dose_response.json")
check_file("5 dose response", OUT / "p3_dose_response.csv", "p3_dose_response.csv")
report("5 dose response")

## Bước 6: Kênh phụ: độ dài tin nhắn

Liệu bản viết lại lọt được vì nó đổi độ dài tin (một kênh phụ mà bộ phát hiện có thể đã dựa vào) chứ không phải vì đổi từ ngữ? Bước này so một bộ phát hiện chỉ
dùng một đặc trưng là độ dài tin với bộ phát hiện n-gram ký tự, cả hai thử trên tin sạch và trên bản viết lại có kiểm soát (20 lần chia).

In [ ]:
def _miss(fit_texts, fit_y, test_texts, kind):
    if kind == "len":
        Xtr = np.array([[len(t)] for t in fit_texts])
        Xte = np.array([[len(t)] for t in test_texts])
    else:
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2)
        Xtr, Xte = v.fit_transform(fit_texts), v.transform(test_texts)
    m = LogisticRegression(max_iter=4000, class_weight="balanced").fit(Xtr, fit_y)
    return 1 - m.predict(Xte).mean()



nf = load_corpus()
nf["para_b"] = nf["id"].map(para_band.pivot(index="src_id", columns="variant", values="text")["b"]).fillna("")
floor = {}
for kind in ("len", "char"):
    clean, attacked = [], []
    for tr, te in StratifiedShuffleSplit(n_splits=SEEDS, test_size=0.3, random_state=0).split(nf, nf.y):
        trd, ted = nf.iloc[tr], nf.iloc[te]
        ph = ted[ted.y == 1]
        atk = ph[ph.para_b.str.len() > 0]
        clean.append(_miss(trd.text.values, trd.y.values, ph.text.values, kind))
        if len(atk):
            attacked.append(_miss(trd.text.values, trd.y.values, atk.para_b.values, kind))
    floor[kind] = (100 * np.mean(clean), 100 * np.mean(attacked))
    print(f"  {kind:5s} miss on clean {floor[kind][0]:.1f}%, on band-attacked {floor[kind][1]:.1f}%")
check_json("6 length channel", {k: {"miss_clean_pct": c, "miss_attacked_pct": a} for k, (c, a) in floor.items()},
           "results/nuisance_floor.json")
report("6 length channel")

## Bước 7: Bỏ một kịch bản ra ngoài

Bộ phát hiện có nhận ra một kịch bản lừa đảo mà nó chưa từng thấy không? Với mỗi kịch bản, mô hình học trên sáu kịch bản còn lại và thử trên kịch bản bị bỏ ra.
Để có mức so sánh, mỗi lần như vậy còn có 20 lần chia ngẫu nhiên giữ đúng số tin lừa đảo và hợp lệ của tập test đó. Khoảng tin cậy lấy bootstrap trên bảy kịch
bản; giá trị p là phép hoán đổi dấu chính xác trên bảy hiệu.

In [ ]:
EXPECTED_SCENARIOS = (
    "bank",
    "delivery",
    "ecommerce",
    "gov",
    "social",
    "tax",
    "telecom",
)


def scenario_folds(df: pd.DataFrame):
    all_idx = np.arange(len(df))
    for scenario in EXPECTED_SCENARIOS:
        test = all_idx[df["scenario"].to_numpy() == scenario]
        train = all_idx[df["scenario"].to_numpy() != scenario]
        train_scenarios = set(df.iloc[train]["scenario"])
        test_scenarios = set(df.iloc[test]["scenario"])
        if train_scenarios & test_scenarios or test_scenarios != {scenario}:
            raise AssertionError(f"scenario leakage in fold {scenario}")
        if set(df.iloc[train]["id"]) & set(df.iloc[test]["id"]):
            raise AssertionError(f"ID leakage in fold {scenario}")
        yield scenario, train, test


def _metrics(y_true: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    ph = y_true == 1
    be = y_true == 0
    return {
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "phishing_miss": float(np.mean(pred[ph] == 0)),
        "benign_fpr": float(np.mean(pred[be] == 1)),
    }


def _fit_predict(text: np.ndarray, y: np.ndarray, train: np.ndarray, test: np.ndarray):
    vectorizer = vec()
    x_train = vectorizer.fit_transform(text[train])
    classifier = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=0)
    classifier.fit(x_train, y[train])
    return classifier.predict(vectorizer.transform(text[test]))


def _matched_random_split(y: np.ndarray, n_benign: int, n_phishing: int, seed: int):
    rng = np.random.default_rng(seed)
    benign = np.flatnonzero(y == 0)
    phishing = np.flatnonzero(y == 1)
    test = np.concatenate(
        [rng.choice(benign, n_benign, replace=False), rng.choice(phishing, n_phishing, replace=False)]
    )
    test = np.sort(test)
    train = np.setdiff1d(np.arange(len(y)), test, assume_unique=True)
    return train, test


def _cluster_bootstrap(values: np.ndarray, seed: int = 260821043, draws: int = 20000):
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(draws, len(values)), replace=True).mean(axis=1)
    return tuple(float(x) for x in np.quantile(samples, [0.025, 0.975]))


def _exact_sign_flip_p(differences: np.ndarray) -> float:
    observed = abs(float(np.mean(differences)))
    null = []
    for signs in itertools.product((-1.0, 1.0), repeat=len(differences)):
        null.append(abs(float(np.mean(differences * np.asarray(signs)))))
    return float(np.mean(np.asarray(null) >= observed - 1e-15))


def loso_evaluate(df: pd.DataFrame):
    text = df["clean_text"].to_numpy()
    y = df["y"].to_numpy(dtype=int)
    rows = []
    oof_true, oof_pred = [], []

    error_rows = []
    for fold_no, (scenario, train, test) in enumerate(scenario_folds(df)):
        pred = _fit_predict(text, y, train, test)
        metric = _metrics(y[test], pred)
        oof_true.extend(y[test])
        oof_pred.extend(pred)
        # keep the misclassified rows themselves, so the error analysis reads this exact run
        for idx, (yt, yp) in zip(test, zip(y[test], pred)):
            if yt != yp:
                error_rows.append({
                    "scenario": scenario,
                    "error": "benign_fp" if yt == 0 else "phishing_miss",
                    "id": df.iloc[idx]["id"],
                    "text": df.iloc[idx]["clean_text"],
                })

        n_benign = int(np.sum(y[test] == 0))
        n_phishing = int(np.sum(y[test] == 1))
        id_scores = []
        for seed in range(SEEDS):
            id_train, id_test = _matched_random_split(
                y, n_benign=n_benign, n_phishing=n_phishing, seed=1000 * fold_no + seed
            )
            id_pred = _fit_predict(text, y, id_train, id_test)
            id_scores.append(_metrics(y[id_test], id_pred)["macro_f1"])

        rows.append(
            {
                "scenario": scenario,
                "n_benign": n_benign,
                "n_phishing": n_phishing,
                **metric,
                "matched_id_macro_f1_mean": float(np.mean(id_scores)),
                "matched_id_macro_f1_std": float(np.std(id_scores, ddof=1)),
                "macro_f1_gap": metric["macro_f1"] - float(np.mean(id_scores)),
                "train_scenarios": ";".join(sorted(set(df.iloc[train]["scenario"]))),
                "test_scenarios": scenario,
            }
        )

    per = pd.DataFrame(rows)
    pooled = _metrics(np.asarray(oof_true), np.asarray(oof_pred))
    ci_lo, ci_hi = _cluster_bootstrap(per["macro_f1"].to_numpy())
    differences = per["macro_f1_gap"].to_numpy()
    summary = pd.DataFrame(
        [
            {
                "n_rows": len(df),
                "n_scenarios": len(per),
                "scenario_balanced_macro_f1": per["macro_f1"].mean(),
                "scenario_balanced_macro_f1_std": per["macro_f1"].std(ddof=1),
                "scenario_bootstrap_ci_low": ci_lo,
                "scenario_bootstrap_ci_high": ci_hi,
                "pooled_oof_macro_f1": pooled["macro_f1"],
                "scenario_balanced_phishing_miss": per["phishing_miss"].mean(),
                "scenario_balanced_benign_fpr": per["benign_fpr"].mean(),
                "matched_id_macro_f1": per["matched_id_macro_f1_mean"].mean(),
                "macro_f1_gap": differences.mean(),
                "exact_sign_flip_p": _exact_sign_flip_p(differences),
            }
        ]
    )
    return per, summary, pd.DataFrame(error_rows)



lo_df = pd.concat([pd.read_csv(DATA / f"synthetic_{t}.csv") for t in ("sms", "email")], ignore_index=True)
lo_df["scenario"] = lo_df["scenario"].fillna("").astype(str).str.strip()
lo_df["clean_text"] = lo_df["text"].fillna("").astype(str).map(strip_url)
lo_df["y"] = (lo_df["label"] == "phishing").astype(int)
lo_df = lo_df[(lo_df["scenario"] != "") & (lo_df["clean_text"].str.len() > 0)].reset_index(drop=True)
assert tuple(sorted(lo_df["scenario"].unique())) == EXPECTED_SCENARIOS and not lo_df["id"].duplicated().any()

per, summary, errors = loso_evaluate(lo_df)
per.to_csv(OUT / "p3_leave_one_scenario_out.csv", index=False)
summary.to_csv(OUT / "p3_leave_one_scenario_out_summary.csv", index=False)
errors.to_csv(OUT / "p3_loso_errors.csv", index=False)
display(per[["scenario", "n_benign", "n_phishing", "macro_f1", "matched_id_macro_f1_mean", "phishing_miss", "benign_fpr"]].round(3))
display(summary.T)
for f in ("p3_leave_one_scenario_out.csv", "p3_leave_one_scenario_out_summary.csv", "p3_loso_errors.csv"):
    check_file("7 leave one scenario out", OUT / f, f)
report("7 leave one scenario out")

## Bước 8: Thử trên tin nhắn SMS thật

Ba bộ phát hiện của Bước 4 (huấn luyện trên corpus mô phỏng, đúng như ở đó, trên từng lần chia) được thử trên một corpus SMS tiếng Việt thật, công khai trên
Hugging Face (CC BY 4.0, ghim revision). Tin trùng nguyên văn bị gộp; link bị xoá và dấu tiếng Việt bị bỏ để chính tả không phân biệt được hai lớp. Để có mốc so,
cùng kiến trúc đó được huấn luyện trên tập train của chính corpus thật và thử trên tập test của nó (bỏ những tin đã có trong train).

In [ ]:
def unaccent(s: str) -> str:
    s = str(s).replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s) if not unicodedata.combining(c))


def prep(texts):
    return [" ".join(unaccent(strip_url(t)).split()) for t in texts]


def sms_head():
    return LogisticRegression(max_iter=3000, class_weight="balanced")


def metrics(y, pred):
    pos, neg = y == 1, y == 0
    return {"miss": float(np.mean(pred[pos] == 0)), "fpr": float(np.mean(pred[neg] == 1)),
            "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0))}



from huggingface_hub import hf_hub_download

REAL_DIR = WORK / "real_sms"
for f in ("full_dataset.csv", "train.csv", "test.csv"):
    hf_hub_download(HF_SMS[0], f, repo_type="dataset", revision=HF_SMS[1], local_dir=REAL_DIR)


def load_real(name):
    d = pd.read_csv(REAL_DIR / name, encoding="utf-8-sig").dropna(subset=["message"])
    return d["message"].astype(str).tolist(), d["label"].astype(int).to_numpy()


raw_t, raw_y = load_real("full_dataset.csv")
seen, t_d, y_d = set(), [], []
for t, y_ in zip(raw_t, raw_y):
    if t not in seen:
        seen.add(t); t_d.append(t); y_d.append(y_)
t_p = prep(t_d)
keep = [i for i, t in enumerate(t_p) if t]
real_x, real_y = [t_p[i] for i in keep], np.array([y_d[i] for i in keep])
print(f"real corpus: {len(raw_t)} rows -> {len(t_d)} distinct -> {len(real_x)} non-empty ({int(real_y.sum())} scam / {int((real_y == 0).sum())} ham)")

txt, y = df_band["text"].to_numpy(), df_band["y"].to_numpy()
pa = df_band["para_a"].to_numpy()
res_real = {d: [] for d in ("D0", "D1", "D2")}
for s in range(SEEDS):
    rng = random.Random(s)
    tr, _te = train_test_split(np.arange(len(df_band)), test_size=0.30, stratify=y, random_state=s)
    ytr = y[tr]
    v0 = vec(); c0 = sms_head().fit(v0.fit_transform(txt[tr]), ytr)
    tr_ph = txt[tr][ytr == 1]
    aug1 = list(txt[tr]) + [perturb(t, rng) for t in tr_ph for _ in range(2)]
    v1 = vec(); c1 = sms_head().fit(v1.fit_transform(aug1), list(ytr) + [1] * (2 * len(tr_ph)))
    aug2 = list(txt[tr]) + [t for t in pa[tr][ytr == 1] if t]
    v2 = vec(); c2 = sms_head().fit(v2.fit_transform(aug2), list(ytr) + [1] * int(sum(1 for t in pa[tr][ytr == 1] if t)))
    for name, (c, v) in (("D0", (c0, v0)), ("D1", (c1, v1)), ("D2", (c2, v2))):
        res_real[name].append(metrics(real_y, c.predict(v.transform(real_x))))

tr_t, tr_y = load_real("train.csv")
te_t, te_y = load_real("test.csv")
tr_p, te_p = prep(tr_t), prep(te_t)
tr_set = set(tr_p)
te_keep = [i for i, t in enumerate(te_p) if t and t not in tr_set]
va = vec()
ca = sms_head().fit(va.fit_transform([t for t in tr_p if t]), tr_y[[i for i, t in enumerate(tr_p) if t]])
anchor = metrics(te_y[te_keep], ca.predict(va.transform([te_p[i] for i in te_keep])))

rows = []
for d, ms in res_real.items():
    for k in ("miss", "fpr", "macro_f1"):
        vals = np.array([m[k] for m in ms])
        rows.append({"detector": d, "metric": k, "mean": vals.mean(), "sd": vals.std()})
for k, v in anchor.items():
    rows.append({"detector": "anchor_real_trained", "metric": k, "mean": v, "sd": 0.0})
pd.DataFrame(rows).to_csv(OUT / "p3_real_sms_check.csv", index=False)
display(pd.DataFrame(rows).pivot(index="detector", columns="metric", values="mean").round(3))
check_json("8 real SMS", {"n_raw": len(raw_t), "n_dedup": len(t_d), "n_empty": len(t_d) - len(keep), "n_real": len(real_x),
                          "n_scam": int(real_y.sum()), "n_ham": int((real_y == 0).sum()), "per_split": res_real,
                          "anchor": anchor, "n_anchor_test": len(te_keep)}, "results/real_sms.json")
check_file("8 real SMS", OUT / "p3_real_sms_check.csv", "p3_real_sms_check.csv")
report("8 real SMS")

## Bước 9: Chấm điểm thủ công các bản viết lại

Hai người đọc 50 cặp (tin gốc, bản viết lại) và trả lời ba câu: bản viết lại giữ nghĩa không, còn là một tin lừa đảo dùng được không, và tin nào thuyết phục hơn
(`A` tin gốc, `B` bản viết lại, `ngang` như nhau). Bước này tính, cho mỗi người, số và tỷ lệ "có" kèm khoảng Wilson; mức đồng thuận giữa hai người (tỷ lệ trùng,
kappa Cohen, và PABAK, vì kappa sụp khi một người chấm "có" gần hết).

In [ ]:
BINARY = ("meaning_preserved", "lure_intact")


PERSUASIVE = ("A", "B", "ngang")


def kappa(x, y, cats):
    n = len(x)
    po = sum(a == b for a, b in zip(x, y)) / n
    pe = sum((x.count(c) / n) * (y.count(c) / n) for c in cats)
    return po, (float("nan") if math.isclose(pe, 1.0) else (po - pe) / (1 - pe))



ratings = pd.read_csv(DATA / "human_eval_ratings.csv", dtype=str)
n = len(ratings)
rated = {w: [(int(r[f"meaning_preserved_{w}"]), int(r[f"lure_intact_{w}"]), r[f"more_persuasive_{w}"]) for _, r in ratings.iterrows()]
         for w in "AB"}
he = {"pairs": n}
for qi, q in enumerate(BINARY):
    for w in "AB":
        k = sum(row[qi] for row in rated[w])
        lo_, hi_ = wilson(k, n)
        he[f"{q}_{w}"] = [k, k / n, lo_, hi_]
    po, kap = kappa([r[qi] for r in rated["A"]], [r[qi] for r in rated["B"]], (0, 1))
    he[f"{q}_agreement"] = [po, kap, 2 * po - 1]
for w in "AB":
    col = [row[2] for row in rated[w]]
    for cat in PERSUASIVE:
        k = col.count(cat)
        he[f"more_persuasive_{cat}_{w}"] = [k, k / n, *wilson(k, n)]
    k = col.count("B") + col.count("ngang")
    he[f"rewrite_at_least_as_persuasive_{w}"] = [k, k / n, *wilson(k, n)]
po, kap = kappa([r[2] for r in rated["A"]], [r[2] for r in rated["B"]], PERSUASIVE)
he["more_persuasive_agreement"] = [po, kap]
for k, v in he.items():
    print(f"  {k:40s} {v}")
check_json("9 human rating", he, "results/human_eval.json")
report("9 human rating")

## Bước 10: Các trang web: thời điểm chụp và tập con cân bằng

Phần thứ hai của bài làm trên trang web. `pages.csv` có 1.919 trang có nội dung tiếng Việt ở cả hai lớp (trang lừa đảo nói tiếng nước ngoài đã bị loại, để bộ
phân loại không học phân biệt ngôn ngữ), mỗi trang có text hiển thị, 21 đặc trưng URL và vector JavaScript nhẹ.

Thời điểm chụp đọc từ mã quét: mã UUID phiên bản 7 mang sẵn thời điểm tạo ở 48 bit đầu. Bước này đếm số trang theo tháng chụp.

Trang hợp lệ nhiều gấp tám lần trang lừa đảo, nên mọi thí nghiệm trên trang dùng một tập con cân bằng 1:1: giữ mọi trang lừa đảo và lấy ngẫu nhiên cùng số trang hợp
lệ, với seed 42 (`balanced_rows`). Tập con và 20 lần chia train/test giống hệt nhau ở mọi bước sau, nên mọi phép so giữa các cấu hình đều là so có cặp.

In [ ]:
TZ = timezone(timedelta(hours=7))


MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def scan_time(uuid: str):
    u = uuid.strip()
    if len(u) != 36 or u[14] != "7":
        return None
    ms = int(u.replace("-", "")[:12], 16)
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).astimezone(TZ)



pages = pd.read_csv(DATA / "pages.csv", keep_default_na=False, na_values=[""], float_precision="round_trip")
pages["text"] = pages["text"].fillna("")
URL_COLS = [c for c in pages.columns if c.startswith("url_")]
JS_COLS = [c for c in pages.columns if c.startswith("js_")]

times = [scan_time(u) for u in pages["scan_uuid"].astype(str)]
dated = [t for t in times if t is not None]
from collections import Counter
by_month = Counter((t.year, t.month) for t in dated)
(y1, m1), n1 = by_month.most_common(1)[0]
(y2, m2), n2 = by_month.most_common(2)[1]
earliest = min(dated)
cw = {"total": len(pages), "peak": [y1, m1, n1], "second": [y2, m2, n2], "tail": len(dated) - n1 - n2,
      "earliest": [earliest.year, earliest.month], "undated": len(times) - len(dated)}
print(f"{len(pages)} pages; {n1} captured in {MONTHS[m1 - 1]} {y1}, {n2} in {MONTHS[m2 - 1]} {y2}, {cw['tail']} earlier "
      f"(from {MONTHS[earliest.month - 1]} {earliest.year}); {cw['undated']} without a capture time")
check_json("10 pages", cw, "results/capture_window.json")


def page_rows():
    # one dict per usable page, in package order: the fields every page experiment reads
    rows = []
    for d in pages[pages.usable == 1].to_dict("records"):
        rows.append({"text": d["text"], "js": [float(d[c]) for c in JS_COLS], "url": [float(d[c]) for c in URL_COLS],
                     "y": 1 if str(d["label"]).lower() in ("1", "phishing", "phish", "smishing", "spam", "malicious") else 0,
                     "tier": str(d["tier"]).strip().lower() if isinstance(d["tier"], str) and d["tier"] else "unassigned",
                     "provenance": str(d["provenance"]).strip().lower() if isinstance(d["provenance"], str) and d["provenance"] else "unknown"})
    return rows


def balanced_rows():
    rows = page_rows()
    rng = random.Random(42)
    ph = [r for r in rows if r["y"] == 1]
    be = [r for r in rows if r["y"] == 0]
    k = min(len(ph), len(be))
    return rng.sample(ph, k) + rng.sample(be, k)


BAL = balanced_rows()
print(f"balanced subset: {sum(r['y'] for r in BAL)} phishing + {sum(1 - r['y'] for r in BAL)} benign pages")
report("10 pages")

## Bước 11: Ghép kênh trên trang web, TF-IDF

Năm cấu hình: chỉ URL, chỉ nội dung (text), chỉ JavaScript, nội dung + URL, và cả ba. Với nội dung, ở bước này là TF-IDF n-gram ký tự (2 đến 5) học lại trên
từng tập train. Các khối đặc trưng được nối ngang rồi đưa vào một hồi quy logistic (`stack`, `make_head`). Mỗi cấu hình chạy trên 20 lần chia 70/30 của tập con
cân bằng; số đo là F1 tại ngưỡng 0,5, PR-AUC, ROC-AUC và tỷ lệ báo nhầm khi bắt được 90% trang lừa đảo (FPR@90%, đọc trên đường cong của tập test).

`text_matrix` là chỗ duy nhất đổi theo bộ mã hoá: với TF-IDF nó học bộ từ vựng trên tập train; với một mô hình ngôn ngữ (Bước 13 trở đi) nó lấy trung bình các
vector ẩn của lớp cuối, và nhớ sẵn vector của mỗi trang vì vector không phụ thuộc cách chia.

In [ ]:
def fpr_at_recall(y, score, target=0.90):
    if len(set(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, score)
    idx = np.where(tpr >= target)[0]
    return float(fpr[idx[0]]) if len(idx) else float("nan")


HF_ENCODERS = {
    "phobert": "vinai/phobert-base",
    "phobert-v2": "vinai/phobert-base-v2",
    "visobert": "uitnlp/visobert",
    "xlm-r": "FacebookAI/xlm-roberta-base",
}


SEGMENTED = {"phobert", "phobert-v2"}  # PhoBERT expects word-segmented input (pyvi)


def _segment(texts):
    try:
        from pyvi import ViTokenizer
        return [ViTokenizer.tokenize(t) for t in texts]
    except ImportError:
        print("[i] pyvi not installed — feeding PhoBERT unsegmented text (slightly weaker).")
        return texts


def text_matrix(train_txt, test_txt, encoder):
    if encoder in HF_ENCODERS:
        try:
            import torch
            tok, mdl = _load_encoder(encoder)

            def emb(texts):
                miss = list(dict.fromkeys(t for t in texts if (encoder, t) not in _EMB_CACHE))
                if miss:
                    seg = _segment(miss) if encoder in SEGMENTED else miss
                    with torch.no_grad():
                        for i in range(0, len(miss), 16):
                            enc = tok([t[:1200] for t in seg[i:i + 16]], truncation=True,
                                      padding=True, max_length=256, return_tensors="pt")
                            vecs = mdl(**enc).last_hidden_state.mean(1).numpy()
                            for t, v in zip(miss[i:i + 16], vecs):
                                _EMB_CACHE[(encoder, t)] = v
                return np.vstack([_EMB_CACHE[(encoder, t)] for t in texts])
            return emb(train_txt), emb(test_txt), "dense"
        except Exception as e:
            print(f"[i] {encoder} unavailable ({type(e).__name__}: {e}); TF-IDF content features.")
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, max_features=30000)
    return vec.fit_transform(train_txt), vec.transform(test_txt), "sparse"


def js_matrix(train_rows, test_rows):
    return (np.array([r["js_emb"] for r in train_rows]),
            np.array([r["js_emb"] for r in test_rows]))


def stack(text_mat, dense_blocks, kind):
    dense = np.hstack(dense_blocks) if dense_blocks else None
    if text_mat is None:
        return dense
    if kind == "sparse":
        return sparse.hstack([text_mat] + ([sparse.csr_matrix(dense)] if dense is not None else [])).tocsr()
    return np.hstack([text_mat] + ([dense] if dense is not None else []))


def make_head(head, seed, ytr):
    if head == "xgboost":
        from xgboost import XGBClassifier
        pos = int(np.sum(ytr == 1))
        return XGBClassifier(tree_method="hist", random_state=seed, n_jobs=-1,
                             eval_metric="logloss",
                             scale_pos_weight=(len(ytr) - pos) / max(pos, 1))
    return LogisticRegression(max_iter=4000, class_weight="balanced")


def evaluate(rows, configs, text_enc, js_enc, use_img, seeds, head="logreg",
             return_scores=False, return_test_metadata=False):
    y = np.array([r["y"] for r in rows])
    idx = np.arange(len(rows))
    precompute_js(rows, js_enc)  # attach r["js_emb"] once (esp. for CodeBERT)
    agg = {c: {m: [] for m in ("F1", "PR-AUC", "ROC-AUC", "FPR@R0.90")} for c in configs}
    curves: dict[str, list] = {c: [] for c in configs}
    for s in range(seeds):
        tr, te = train_test_split(idx, test_size=0.30, stratify=y, random_state=s)
        trr = [rows[i] for i in tr]
        ter = [rows[i] for i in te]
        ytr, yte = y[tr], y[te]
        # precompute each modality once per split
        tmat_tr, tmat_te, kind = text_matrix([r["text"] for r in trr], [r["text"] for r in ter], text_enc)
        js_tr, js_te = js_matrix(trr, ter)
        url_tr = np.array([r["url"] for r in trr]); url_te = np.array([r["url"] for r in ter])
        img_tr = img_te = None
        if use_img:
            img_tr, img_te = image_block(trr, ter)

        for cfg in configs:
            txt = (tmat_tr, tmat_te, kind) if "content" in cfg else (None, None, "dense")
            blocks_tr, blocks_te = [], []
            if "url" in cfg:
                blocks_tr.append(url_tr); blocks_te.append(url_te)
            if "js" in cfg:
                blocks_tr.append(js_tr); blocks_te.append(js_te)
            if "image" in cfg and img_tr is not None:
                blocks_tr.append(img_tr); blocks_te.append(img_te)
            Xtr = stack(txt[0], blocks_tr, txt[2])
            Xte = stack(txt[1], blocks_te, txt[2])
            if Xtr is None:
                continue
            clf = make_head(head, s, ytr)
            clf.fit(Xtr, ytr)
            sc = clf.predict_proba(Xte)[:, 1]
            pred = (sc >= 0.5).astype(int)
            agg[cfg]["F1"].append(f1_score(yte, pred, zero_division=0))
            agg[cfg]["PR-AUC"].append(average_precision_score(yte, sc))
            agg[cfg]["ROC-AUC"].append(roc_auc_score(yte, sc) if len(set(yte)) > 1 else float("nan"))
            agg[cfg]["FPR@R0.90"].append(fpr_at_recall(yte, sc))
            if return_scores:
                if return_test_metadata:
                    curves[cfg].append((yte.copy(), sc.copy(), {
                        "tier": np.asarray([rows[i].get("tier", "unassigned") for i in te]),
                        "provenance": np.asarray([rows[i].get("provenance", "unknown") for i in te]),
                    }))
                else:
                    curves[cfg].append((yte.copy(), sc.copy()))
    return (agg, curves) if return_scores else agg



_EMB_CACHE = {}
MODEL_CACHE = {}


def _load_encoder(encoder):
    # filled in at step 13; until then only TF-IDF is used
    return MODEL_CACHE[encoder]


def precompute_js(rows, encoder):
    # the lightweight JavaScript vector is used as is
    for r in rows:
        r["js_emb"] = np.asarray(r["js"], dtype=float)


CONFIGS = ["url", "content", "js", "content+url", "content+url+js"]
FUSION_SEEDS = 20
FUSION = {}


def fusion_table(encoder, head="logreg"):
    agg = evaluate(BAL, CONFIGS, encoder, "lightweight", False, FUSION_SEEDS, head=head)
    n_ph = sum(r["y"] for r in BAL)
    out = {"per_split": agg}
    if head == "logreg":
        def paired(a, b, metric):
            s = corrected_paired_t(np.asarray(agg[a][metric], float) - np.asarray(agg[b][metric], float))
            return s["mean"], s["p"], s["wins"], s["k"], s["p_naive"]
        out = {"n_phishing": n_ph, "n_benign": n_ph, "per_split": agg,
               "fusion_minus_content": {m: paired("content+url", "content", m) for m in ("F1", "PR-AUC", "ROC-AUC")},
               "fusion_minus_url_f1": paired("content+url", "url", "F1")}
    FUSION[f"{encoder}|{head}"] = out
    print(pd.DataFrame({c: {m: f"{np.mean(v):.3f} +- {np.std(v):.3f}" for m, v in agg[c].items()} for c in CONFIGS}).T)
    return out


tf = fusion_table("tfidf")
for m, (d, p, w, k, _) in tf["fusion_minus_content"].items():
    print(f"content+URL minus content, {m}: {d:+.3f} (corrected p={p:.3f}, {w}/{k} splits)")
d, p, w, k, _ = tf["fusion_minus_url_f1"]
print(f"content+URL minus URL, F1: {d:+.3f} (corrected p={p:.3g}, {w}/{k} splits)")
check_json("11 TF-IDF fusion", tf, "results/content_fusion.json", key="tfidf|logreg")
report("11 TF-IDF fusion")

## Bước 12: Độ nhạy theo tầng nhãn

Nhãn của các trang lừa đảo có ba tầng bằng chứng (`bronze`, `silver`, `gold`), cộng nhóm chưa xếp tầng. Bước này lặp lại ba cấu hình của Bước 11 (URL, nội dung,
nội dung + URL; TF-IDF, cùng 20 lần chia) và ghi lại, cho mỗi lần chia, F1 chung và tỷ lệ bắt được trang lừa đảo trong từng tầng. Nếu kết luận chỉ đứng được nhờ
các nhãn yếu, cột theo tầng sẽ lộ ra.

In [ ]:
cfg3 = ["url", "content", "content+url"]
_, curves = evaluate(BAL, cfg3, "tfidf", "lightweight", False, 20, return_scores=True, return_test_metadata=True)
records = []
for config in cfg3:
    for seed, (yy, score, meta) in enumerate(curves[config]):
        pred = score >= 0.5
        records.append({"config": config, "seed": seed, "tier": "overall", "n": len(yy), "metric": "F1", "value": f1_score(yy, pred)})
        for tier in ("bronze", "silver", "gold", "unassigned"):
            mask = (yy == 1) & (meta["tier"] == tier)
            records.append({"config": config, "seed": seed, "tier": tier, "n": int(mask.sum()), "metric": "recall",
                            "value": float(pred[mask].mean()) if mask.any() else np.nan})
tier_df = pd.DataFrame(records)
tier_df.to_csv(OUT / "p3_tier_sensitivity.csv", index=False, float_format="%.6f")
display(tier_df.groupby(["config", "tier"])["value"].mean().unstack().round(3))
check_file("12 label tiers", OUT / "p3_tier_sensitivity.csv", "p3_tier_sensitivity.csv")
report("12 label tiers")

## Bước 13: Bộ mã hoá văn bản: tải và nạp

Bốn mô hình được tải từ Hugging Face, mỗi mô hình ghim theo một revision:

| Mô hình | Kho | Ghi chú |
|---|---|---|
| PhoBERT | `vinai/phobert-base` | đơn ngữ tiếng Việt; cần tách từ trước (`pyvi`: "ngân hàng" thành "ngân_hàng") |
| PhoBERT-v2 | `vinai/phobert-base-v2` | như trên, huấn luyện trên nhiều dữ liệu hơn |
| ViSoBERT | `uitnlp/visobert` | đơn ngữ, huấn luyện trên mạng xã hội tiếng Việt |
| XLM-R | `FacebookAI/xlm-roberta-base` | đa ngữ |

Ba mô hình đầu chỉ phát hành trọng số dạng `pytorch_model.bin`. Ô dưới đọc file đó bằng `torch.load(weights_only=True)` (chỉ nạp tensor, không chạy mã) rồi nạp
vào kiến trúc dựng từ `config.json`, bỏ tiền tố `roberta.` của tên tham số và kiểm rằng chỉ tầng pooler (không dùng) là thiếu, và thứ duy nhất thừa là
`position_ids` (một dãy chỉ số mà mô hình tự dựng lại). XLM-R có sẵn `model.safetensors`.
Mô hình chạy ở chế độ suy luận, không tinh chỉnh.

In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModel, AutoTokenizer

ENCODER_REVS = {
    "phobert": ("vinai/phobert-base", "01daacda68afe13d83023d16ec647239e344a1e6"),
    "phobert-v2": ("vinai/phobert-base-v2", "86cd7fd4c148980922ac11a2cf5e257f2ba639e1"),
    "visobert": ("uitnlp/visobert", "196a62afad9cbe4f52a54aabad828b13f0eec59a"),
    "xlm-r": ("FacebookAI/xlm-roberta-base", "e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"),
}


def load_encoder(encoder):
    repo, rev = ENCODER_REVS[encoder]
    bin_only = encoder != "xlm-r"
    weights = "pytorch_model.bin" if bin_only else "model.safetensors"
    d = snapshot_download(repo, revision=rev, allow_patterns=["*.json", "*.txt", "*.codes", "*.model", weights])
    tok = AutoTokenizer.from_pretrained(d, use_fast=encoder not in SEGMENTED)
    if not bin_only:
        return tok, AutoModel.from_pretrained(d).eval()
    mdl = AutoModel.from_config(AutoConfig.from_pretrained(d))
    state = torch.load(os.path.join(d, weights), map_location="cpu", weights_only=True)
    state = {k[len("roberta."):]: v for k, v in state.items() if k.startswith("roberta.")}
    missing, unexpected = mdl.load_state_dict(state, strict=False)
    # the pooler is not used; position_ids is a buffer the model rebuilds itself
    assert set(unexpected) <= {"embeddings.position_ids"} and all(k.startswith("pooler.") for k in missing), (missing, unexpected)
    return tok, mdl.eval()


for enc in ENCODER_REVS:
    MODEL_CACHE[enc] = load_encoder(enc)
    print(f"{enc}: {sum(p.numel() for p in MODEL_CACHE[enc][1].parameters()) / 1e6:.0f}M parameters")

## Bước 14: Quét năm bộ mã hoá

Cùng tập con, cùng 20 lần chia, ba cấu hình (URL, nội dung, nội dung + URL) cho mỗi bộ mã hoá nội dung. File `p3_encoder_sweep.csv` lưu từng số đo của từng lần chia;
từ đó, cho mỗi bộ mã hoá và mỗi số đo, hai phép so có cặp: thêm URL vào nội dung (`uc`), và thêm nội dung vào URL (`cu`). Giá trị q là p đã hiệu chỉnh BH trong từng
cột (một câu hỏi, năm bộ mã hoá); `q_all` hiệu chỉnh trên cả 40 phép so.

Bước này chạy lâu nhất: mỗi trang được mã hoá một lần cho mỗi mô hình, rồi 60 lần huấn luyện hồi quy logistic trên vector 768 chiều.

In [ ]:
SWEEP_ENCODERS = ("tfidf", "phobert", "phobert-v2", "visobert", "xlm-r")
SWEEP_METRICS = ("F1", "PR-AUC", "ROC-AUC", "FPR@R0.90")
SWEEP_DIRECTIONS = (("uc", ("content+url", "content")), ("cu", ("content+url", "url")))

recs, scores = [], {}
for enc in SWEEP_ENCODERS:
    a = evaluate(BAL, ["url", "content", "content+url"], enc, "lightweight", False, FUSION_SEEDS)
    scores[enc] = a
    for cfg, mets in a.items():
        for metric, vals in mets.items():
            for i, v in enumerate(vals):
                recs.append({"encoder": enc, "config": cfg, "metric": metric, "split": i, "value": f"{float(v):.6f}"})
    print(f"{enc}: content F1 {np.mean(a['content']['F1']):.3f}, content+URL F1 {np.mean(a['content+url']['F1']):.3f}", flush=True)
with open(OUT / "p3_encoder_sweep.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["encoder", "config", "metric", "split", "value"])
    w.writeheader(); w.writerows(recs)
check_file("14 encoder sweep", OUT / "p3_encoder_sweep.csv", "p3_encoder_sweep.csv")

# the tests are run on the 6-decimal values the CSV holds, exactly as the paper's tables are
cached = {}
for r in recs:
    cached.setdefault(r["encoder"], {}).setdefault(r["config"], {}).setdefault(r["metric"], []).append(float(r["value"]))
verdicts = []
for enc in SWEEP_ENCODERS:
    agg = cached[enc]
    rec = {"enc": enc, "mean": {cfg: {m: float(np.mean(agg[cfg][m])) for m in SWEEP_METRICS} for cfg in ("url", "content", "content+url")}}
    for direction_, (a, b) in SWEEP_DIRECTIONS:
        for metric in SWEEP_METRICS:
            rec[(direction_, metric)] = corrected_paired_t(np.asarray(agg[a][metric], float) - np.asarray(agg[b][metric], float))
    verdicts.append(rec)
k = len(verdicts)
cols = [(d, m) for d, _ in SWEEP_DIRECTIONS for m in SWEEP_METRICS]
for col in cols:
    q, _ = bh_adjust([v[col]["p"] for v in verdicts])
    for v, qi in zip(verdicts, q):
        v[col]["q"] = qi
q_all, _ = bh_adjust([v[col]["p"] for col in cols for v in verdicts])
for i, col in enumerate(cols):
    for j, v in enumerate(verdicts):
        v[col]["q_all"] = q_all[i * k + j]
display(pd.DataFrame({v["enc"]: {f"{d} {m}": f"{v[(d, m)]['mean']:+.3f} (q={v[(d, m)]['q']:.3f})" for d, m in cols} for v in verdicts}))
check_json("14 encoder sweep", {v["enc"]: {("mean" if c == "mean" else "|".join(c)): v[c] for c in v if c != "enc"} for v in verdicts},
           "results/encoder_sweep.json")
report("14 encoder sweep")

## Bước 15: Kiểm định tương đương (TOST)

Bước 14 không tìm thấy hiệu dương có ý nghĩa của việc thêm URL vào nội dung, nhưng "không có ý nghĩa" không có nghĩa là "bằng nhau". Phép TOST (hai phép kiểm một
phía, mỗi phía α = 0,05, cùng phương sai hiệu chỉnh) kiểm xem hiệu F1 có nằm trọn trong ±δ hay không, với hai biên δ = 0,02 và 0,05 được khai báo trước, không suy
ra từ dữ liệu. Giá trị p của TOST là giá trị lớn hơn trong hai phía.

In [ ]:
TEST_FRAC = 0.30


MARGINS = (0.02, 0.05)


def tost(d, margin):
    k = len(d)
    mean = float(np.mean(d))
    var = float(np.var(d, ddof=1))
    se = math.sqrt((1.0 / k + TEST_FRAC / (1.0 - TEST_FRAC)) * var)
    t_lo = (mean + margin) / se   # H0: mean <= -margin
    t_hi = (mean - margin) / se   # H0: mean >= +margin
    p = max(float(stats.t.sf(t_lo, k - 1)), float(stats.t.cdf(t_hi, k - 1)))
    return mean, se, p



tost_res = {}
for enc in SWEEP_ENCODERS:
    d = np.array(cached[enc]["content+url"]["F1"]) - np.array(cached[enc]["content"]["F1"])
    cells_ = [(m, *tost(d, m)[::2]) for m in MARGINS]
    tost_res[enc] = {"mean_diff": float(np.mean(d)), **{f"p_tost_{m}": p for m, _, p in cells_}}
    print(f"{enc:11s} mean diff {np.mean(d):+.4f}  " + "  ".join(f"delta={m:.2f}: p={p:.3f}{' (equivalent)' if p < 0.05 else ''}" for m, _, p in cells_))
check_json("15 TOST", tost_res, "results/tost.json")
report("15 TOST")

## Bước 16: XLM-R với đầu tuyến tính và đầu XGBoost

Bảng của Bước 11 làm lại với XLM-R thay cho TF-IDF (cùng năm cấu hình), rồi làm lại lần nữa với đầu phân loại XGBoost thay cho hồi quy logistic, trên đúng các khối
đặc trưng và lần chia đó (`make_head`). Đây là kiểu ghép "transformer rồi cây tăng cường" thường gặp trong các bài lai; vì cùng lần chia, mọi hiệu giữa hai đầu là
hiệu có cặp.

In [ ]:
xl = fusion_table("xlm-r")
xg = fusion_table("xlm-r", head="xgboost")
for (a, b, metric) in (("content+url", "content", "F1"), ("content+url", "content", "PR-AUC"), ("content+url+js", "content+url", "F1")):
    s = corrected_paired_t(np.asarray(xg["per_split"][a][metric], float) - np.asarray(xg["per_split"][b][metric], float))
    print(f"XGBoost head, {a} minus {b}, {metric}: {s['mean']:+.3f} (corrected p={s['p']:.3f})")
for cfg in ("content+url", "content"):
    s = corrected_paired_t(np.asarray(xg["per_split"][cfg]["F1"], float) - np.asarray(xl["per_split"][cfg]["F1"], float))
    print(f"{cfg}: XGBoost head minus linear head, F1 {s['mean']:+.3f} (corrected p={s['p']:.3f})")
check_json("16 XLM-R heads", FUSION, "results/content_fusion.json")
report("16 XLM-R heads")

## Bước 17: Chuẩn hoá thang đo và ghép muộn

Đặc trưng URL là các số đếm thô, thang đo khác hẳn đặc trưng nội dung. Liệu việc ghép ba kênh kém hơn chỉ vì thang đo? Phép thử bổ sung này (TF-IDF và XLM-R, cùng
tập con và lần chia) chạy mỗi cấu hình theo ba cách: nối thô (`raw`), nối sau khi chuẩn hoá từng khối trên tập train (`std`), và ghép muộn: mỗi kênh một mô hình,
xác suất ngoài mẫu của chúng (5-fold) làm đầu vào cho một mô hình thứ hai (`late`). Các phép so ở đây là post hoc và không hiệu chỉnh BH.

In [ ]:
STD_CONFIGS = ("url", "content", "content+url", "content+url+js")


def std_fpr_at_recall(y, sc, target=0.90):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y, sc)
    ok = tpr >= target
    return float(fpr[ok][0]) if ok.any() else 1.0


def std_head():
    return LogisticRegression(max_iter=4000, class_weight="balanced")


def early_fit(txt, blocks_tr, blocks_te, kind, ytr, standardise):
    if standardise:
        std = []
        for btr, bte in zip(blocks_tr, blocks_te):
            sc = StandardScaler().fit(btr)
            std.append((sc.transform(btr), sc.transform(bte)))
        blocks_tr = [a for a, _ in std]
        blocks_te = [b for _, b in std]
        if kind == "dense" and txt[0] is not None:
            sc = StandardScaler().fit(txt[0])
            txt = (sc.transform(txt[0]), sc.transform(txt[1]), kind)
    Xtr = tcf.stack(txt[0], blocks_tr, kind)
    Xte = tcf.stack(txt[1], blocks_te, kind)
    clf = std_head().fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1]


def late_fit(channels, ytr):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    n = len(ytr)
    meta_tr = np.zeros((n, len(channels)))
    meta_te = np.zeros((channels[0][1].shape[0], len(channels)))
    for j, (Xtr, Xte) in enumerate(channels):
        oof = np.zeros(n)
        for tr_i, va_i in kf.split(np.zeros(n), ytr):
            m = std_head().fit(Xtr[tr_i], ytr[tr_i])
            oof[va_i] = m.predict_proba(Xtr[va_i])[:, 1]
        meta_tr[:, j] = oof
        meta_te[:, j] = std_head().fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    meta = std_head().fit(meta_tr, ytr)
    return meta.predict_proba(meta_te)[:, 1]


def std_run(encoder):
    rows = std_balanced_rows(encoder)
    y = np.array([r["y"] for r in rows])
    idx = np.arange(len(rows))
    res = {}  # (variant, cfg) -> {"F1": [...], "FPR": [...]}

    def rec(variant, cfg, yte, sc):
        d = res.setdefault((variant, cfg), {"F1": [], "FPR": []})
        d["F1"].append(f1_score(yte, (sc >= 0.5).astype(int), zero_division=0))
        d["FPR"].append(std_fpr_at_recall(yte, sc))

    for s in range(SEEDS):
        tr, te = train_test_split(idx, test_size=0.30, stratify=y, random_state=s)
        trr = [rows[i] for i in tr]
        ter = [rows[i] for i in te]
        ytr, yte = y[tr], y[te]
        tmat_tr, tmat_te, kind = tcf.text_matrix(
            [r["text"] for r in trr], [r["text"] for r in ter], encoder)
        js_tr, js_te = tcf.js_matrix(trr, ter)
        url_tr = np.array([r["url"] for r in trr])
        url_te = np.array([r["url"] for r in ter])

        def parts(cfg):
            txt = (tmat_tr, tmat_te, kind) if "content" in cfg else (None, None, kind)
            btr, bte = [], []
            if "url" in cfg:
                btr.append(url_tr); bte.append(url_te)
            if "js" in cfg:
                btr.append(js_tr); bte.append(js_te)
            return txt, btr, bte

        for cfg in STD_CONFIGS:
            txt, btr, bte = parts(cfg)
            rec("raw", cfg, yte, early_fit(txt, btr, bte, kind, ytr, standardise=False))
            rec("std", cfg, yte, early_fit(txt, btr, bte, kind, ytr, standardise=True))
            if "+" in cfg:  # late fusion is defined for multi-channel configs only
                chans = []
                if "content" in cfg:
                    if kind == "sparse":
                        chans.append((tmat_tr.tocsr(), tmat_te.tocsr()))
                    else:
                        chans.append((tmat_tr, tmat_te))
                if "url" in cfg:
                    chans.append((url_tr, url_te))
                if "js" in cfg:
                    chans.append((js_tr, js_te))
                rec("late", cfg, yte, late_fit(chans, ytr))
    return res



tcf = SimpleNamespace(text_matrix=text_matrix, js_matrix=js_matrix, stack=stack)


def std_balanced_rows(encoder):
    rows = balanced_rows()
    precompute_js(rows, "lightweight")
    return rows


all_rows, split_rows, std_results = [], [], {}
any_std_gain = False
for enc in ("tfidf", "xlm-r"):
    res = std_run(enc)
    for (variant, cfg), d in res.items():
        all_rows.append({"encoder": enc, "variant": variant, "config": cfg, "f1_mean": np.mean(d["F1"]), "f1_std": np.std(d["F1"]),
                         "fpr90_mean": np.mean(d["FPR"]), "fpr90_std": np.std(d["FPR"])})
    stk = "content+url+js"
    t_std = corrected_paired_t(np.array(res[("std", stk)]["F1"]) - np.array(res[("raw", stk)]["F1"]))
    t_late = corrected_paired_t(np.array(res[("late", stk)]["F1"]) - np.array(res[("raw", stk)]["F1"]))
    t_vs_url = corrected_paired_t(np.array(res[("std", stk)]["F1"]) - np.array(res[("raw", "url")]["F1"]))
    any_std_gain |= (t_std["mean"] > 0 and t_std["p"] < 0.05) or (t_late["mean"] > 0 and t_late["p"] < 0.05)
    head_c = {}
    for variant in ("raw", "std", "late"):
        a = np.array(res[(variant, "content+url")]["F1"])
        b = np.array(res[("raw" if variant == "late" else variant, "content")]["F1"])
        head_c[variant] = corrected_paired_t(a - b)
        for metric in ("F1", "FPR"):
            for i, v in enumerate(res[(variant, "content+url")][metric]):
                split_rows.append({"encoder": enc, "variant": variant, "config": "content+url", "metric": metric, "split": i, "value": v})
    for i, v in enumerate(res[("raw", "content")]["F1"]):
        split_rows.append({"encoder": enc, "variant": "raw", "config": "content", "metric": "F1", "split": i, "value": v})
    std_results[enc] = {"per_split": {f"{v}|{c}": d for (v, c), d in res.items()}, "std_vs_raw_stack": t_std,
                        "late_vs_raw_stack": t_late, "std_stack_vs_url": t_vs_url, "headline": head_c}
    print(f"{enc}: content+URL minus content, F1: " + ", ".join(f"{v} {head_c[v]['mean']:+.3f} (p={head_c[v]['p']:.3f})" for v in head_c))

with open(OUT / "p3_standardization_probe.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(all_rows[0].keys())); w.writeheader(); w.writerows(all_rows)
with open(OUT / "p3_standardization_probe_splits.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(split_rows[0].keys())); w.writeheader(); w.writerows(split_rows)
check_file("17 scaling and late fusion", OUT / "p3_standardization_probe.csv", "p3_standardization_probe.csv")
check_file("17 scaling and late fusion", OUT / "p3_standardization_probe_splits.csv", "p3_standardization_probe_splits.csv")
check_json("17 scaling and late fusion", {"encoders": std_results, "any_std_gain": any_std_gain}, "results/standardization.json")
report("17 scaling and late fusion")

## Bước 18: Tổng hợp

Bảng dưới gom mọi phép kiểm theo bước.

In [ ]:
tbl = pd.DataFrame(CHECKS)
display(tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum"))
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")